# HALO — Feature Extraction on Colab
**What this notebook does:**
1. Clones the HALO repo
2. Installs dependencies
3. Downloads Something-Something-v2 (you need your own credentials)
4. Runs the frozen V-JEPA encoder over all videos and saves `.pt` feature files
5. Zips and saves to Google Drive

> Make sure you select **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# ── 1. Check GPU ─────────────────────────────────────────────────────────────
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 2. Clone HALO repo ───────────────────────────────────────────────────────
!git clone https://github.com/Ojaswa-Yadav/HALO.git
%cd HALO

In [ ]:
# ── 3. Install dependencies ──────────────────────────────────────────────────
!pip install -q torch torchvision einops timm omegaconf decord tqdm rich

In [ ]:
# ── 4. Mount Google Drive (features will be saved here) ──────────────────────
from google.colab import drive
drive.mount('/content/drive')

FEAT_DIR = '/content/drive/MyDrive/HALO/ssv2_feats'
!mkdir -p {FEAT_DIR}/train
!mkdir -p {FEAT_DIR}/validation
print(f"Features will be saved to: {FEAT_DIR}")

In [ ]:
# ── 5. Download Something-Something-v2 ───────────────────────────────────────
# You need to get download links from: https://developer.qualcomm.com/software/ai-datasets/something-something
# After registering, paste the download commands below.
#
# The dataset comes as 20 zip parts (~1GB each). Example:
#   wget -c "<YOUR_LINK>/20bn-something-something-v2-00" -O /content/ssv2/20bn-something-something-v2-00
#
# For now we set up the directory structure:

!mkdir -p /content/ssv2/videos
!mkdir -p /content/ssv2/labels

# PASTE YOUR WGET COMMANDS HERE:
# !wget -c "<link>" -O /content/ssv2/20bn-something-something-v2-00
# ... (repeat for all 20 parts)

print("Paste your wget commands in this cell and re-run.")

In [ ]:
# ── 5b. Unzip all parts ──────────────────────────────────────────────────────
# Run after downloading all parts
import os, glob

parts = sorted(glob.glob('/content/ssv2/20bn-something-something-v2-*'))
print(f"Found {len(parts)} parts")

for part in parts:
    print(f"Extracting {part}...")
    os.system(f'cat {part} | tar zx -C /content/ssv2/videos/')

print("Done. Total videos:")
!ls /content/ssv2/videos/ | wc -l

In [ ]:
# ── 5c. Download label files ─────────────────────────────────────────────────
# Labels are in a separate download from the same registration page
# !wget -c "<labels_link>" -O /content/ssv2/labels.zip
# !unzip /content/ssv2/labels.zip -d /content/ssv2/labels/

# Verify structure
!ls /content/ssv2/labels/

In [ ]:
# ── 6. Update config to point to Colab paths ─────────────────────────────────
from omegaconf import OmegaConf

cfg = OmegaConf.load('configs/base.yaml')
cfg.data.root       = '/content/ssv2'
cfg.data.feat_cache = FEAT_DIR
cfg.data.num_workers = 2   # Colab works better with fewer workers

print(OmegaConf.to_yaml(cfg))

In [ ]:
# ── 7. Run feature extraction ─────────────────────────────────────────────────
# This is the main step — runs frozen V-JEPA over all videos
# T4: ~10-12 hours for full SSv2. Run overnight or use Colab Pro.
# Saves one .pt file per video to Google Drive (so progress is safe if session dies)

import sys
sys.path.insert(0, '.')

from scripts.extract_features import extract

extract(cfg, split='train')
extract(cfg, split='validation')

In [ ]:
# ── 8. Verify output ─────────────────────────────────────────────────────────
import torch, os

train_feats = os.listdir(f'{FEAT_DIR}/train')
print(f"Train feature files: {len(train_feats)}")

# spot check one file
sample = torch.load(f'{FEAT_DIR}/train/{train_feats[0]}')
print(f"Sample shape: {sample.shape}")  # should be (T, N, D) e.g. (16, 196, 768)
print(f"Approx total size: {len(train_feats) * sample.element_size() * sample.numel() / 1e6:.1f} MB per video")